## PydanticAI 이미지 데이터 정형화

- [PydanticAI 이미지/문서 입력 공식 문서](https://ai.pydantic.dev/input/#image-input)
- [Gemini API 이미지 이해 문서](https://ai.google.dev/gemini-api/docs/vision)

**학습 흐름**

이 노트북은 앞서 배운 두 가지 기능을 **결합** 합니다.

| 노트북 | 배운 내용 | 이 노트북에서 활용 |
|--------|----------|------------------|
| 5번 - 이미지 이해 | `BinaryContent`로 이미지 전달 | 패션/영수증 이미지를 Agent에 입력 |
| 2번 - 구조화된 출력 | `output_type`으로 Pydantic 스키마 강제 | 이미지 분석 결과를 정형 데이터로 변환 |

**실습 목표**
- 패션 이미지 자동 분류 및 메타데이터 추출
- 영수증 이미지 자동 입력 및 회계 데이터화
- 이미지 => 정형 데이터 변환 프로세스 이해
- 분석 결과 시각화 및 데이터 품질 검증

**데이터셋 출처**
- 패션: http://mmlab.ie.cuhk.edu.hk/projects/DeepFashion.html
- 영수증: https://github.com/clovaai/cord

In [ ]:
# ========================================
# 필요한 라이브러리를 불러옵니다
# ========================================

import os
import time
import platform
from pprint import pprint
from pathlib import Path
from typing import List, Literal, Optional
from datetime import datetime

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from pydantic_ai import Agent, BinaryContent
from pydantic_ai.models.google import GoogleModelSettings

# 운영체제별 한글 폰트 설정
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
else:
    plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

# .env 파일에서 API 키 로드
load_dotenv()
api_key = os.getenv('GEMINI_API_KEY')
gemini_model = os.getenv('GEMINI_MODEL', 'gemini-3.1-flash-lite-preview')

# PydanticAI는 GEMINI_API_KEY 환경변수를 자동으로 인식합니다
# 모델 ID 형식: 'google-gla:{모델명}'
model_id = f'google-gla:{gemini_model}'

# API 키 유효성 검사
api_key_valid = api_key and 'YOUR_API_KEY' not in api_key
print(f"API 키 설정 확인: {'✓' if api_key_valid else '✗'}")
if not api_key_valid:
    print("  .env 파일에서 GEMINI_API_KEY를 실제 API 키로 설정해주세요!")
print(f"모델 확인: {model_id}")

# 데이터 폴더 경로 설정
DATA_DIR = Path("data")
FASHION_DIR = DATA_DIR / "fashion"
RECEIPT_DIR = DATA_DIR / "receipt"
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

print()
print(f"데이터 폴더 확인:")
print(f"  패션 이미지 폴더: {FASHION_DIR.exists()} - {FASHION_DIR}")
print(f"  영수증 이미지 폴더: {RECEIPT_DIR.exists()} - {RECEIPT_DIR}")
print(f"  출력 폴더: {OUTPUT_DIR} (생성 완료)")

# API 호출 간격 (초)
API_DELAY = 3


def load_image_content(image_path):
    """이미지 파일을 BinaryContent로 변환하는 헬퍼 함수"""
    path = Path(image_path)

    # 파일 확장자로 media_type 결정
    suffix = path.suffix.lower()
    media_types = {
        '.jpg': 'image/jpeg',
        '.jpeg': 'image/jpeg',
        '.png': 'image/png',
        '.webp': 'image/webp',
    }
    media_type = media_types.get(suffix, 'image/jpeg')

    # 이미지 바이트 읽기
    with open(path, 'rb') as f:
        image_bytes = f.read()

    # BinaryContent로 변환
    return BinaryContent(data=image_bytes, media_type=media_type)

### 1. 패션 이미지 분석 및 정형화

**비즈니스 목표**
- 대량의 패션 상품 이미지를 자동으로 분류하고 메타데이터를 추출합니다.
- 수작업 태깅 시간을 95% 단축합니다.
- 일관된 상품 정보를 관리합니다.

**활용 사례**
- 이커머스 상품 등록 자동화
- 재고 관리 시스템 연동
- 검색/추천 시스템 데이터 구축

#### 1-1. 패션 이미지 분석 스키마 설계

**패션 이미지 분석 스키마 설계 프로세스**

**1. 비즈니스 질문부터 시작**

**Q: 패션 이미지로 무엇을 얻고 싶은가?**

A: 패션 이미지를 분석해서 성별, 의류 아이템별 상세 정보, 전체 스타일, 계절 적합성을 자동으로 파악하고 싶습니다.

**2. 필요한 정보 나열**

- **성별 구분**: 남성용? 여성용? 남녀공용?
- **의류 아이템 정보**:
  - 어떤 종류? (상의/하의/아우터 등)
  - 구체적 명칭? (화이트 반팔 티셔츠)
  - 색상? (최대 2개 주요 색상)
  - 패턴? (스트라이프/체크/무지)
  - 스타일? (캐주얼/포멀/스트릿)
  - 소재? (코튼/데님/니트)
- **전체 코디 스타일**: 캐주얼? 포멀? 스트릿? 미니멀?
- **계절 적합성**: 봄/여름/가을/겨울/사계절
- **종합 설명**: 전체 코디나 상품에 대한 간단한 설명문

**3. 데이터 타입 결정**

| 정보 | 타입 | 이유 |
|------|------|------|
| 성별 | `Literal["male", "female", "unisex"]` | 3가지 고정 카테고리 |
| 의류 타입 | `Literal["top", "bottom", ...]` | 6가지 고정 카테고리 |
| 스타일 | `Literal["casual", "formal", ...]` | 6가지 스타일 중 선택 |
| 계절 | `Literal["spring", "summer", ...]` | 5가지 계절 옵션 |
| 아이템 목록 | `List[ClothingItem]` | 1~5개 아이템 (중첩 구조) |
| 색상 | `List[str]` | 최대 2개 주요 색상 |
| 패턴 | `Optional[str]` | 패턴이 없을 수도 있음 |
| 소재 | `Optional[str]` | 추정 불가능할 수도 있음 |
| 설명 | `str` | 100자 이내 텍스트 |

선택지가 고정된 필드에는 `Literal`을 사용합니다. `description`에 각 값의 의미를 명시하면 LLM이 정확하게 분류합니다.

**실습 (1)**
- 패션 이미지 분석을 위한 Pydantic 스키마를 정의합니다.
- `Literal`로 고정 카테고리를 표준화하고, `ClothingItem` 중첩 구조로 개별 아이템 정보를 관리합니다.

In [ ]:
# =============================================================================
# 패션 이미지 분석 스키마 정의
# =============================================================================

class ClothingItem(BaseModel):
    """개별 의류 아이템 정보 (중첩 구조)"""
    item_type: Literal["top", "bottom", "dress", "outerwear", "shoes", "accessories"] = Field(
        description="아이템 타입. top: 상의(티셔츠/셔츠/니트/후드), bottom: 하의(바지/스커트), dress: 원피스/드레스, outerwear: 아우터(재킷/코트/패딩), shoes: 신발, accessories: 악세서리(가방/모자/벨트)"
    )
    detailed_name: str = Field(
        description="상세 명칭 (예: '화이트 반팔 티셔츠', '블랙 스키니 진')"
    )
    colors: List[str] = Field(
        description="주요 색상 한글 표기, 많은 순서대로 (예: ['블랙', '화이트'])",
        max_length=2
    )
    pattern: Optional[str] = Field(
        default=None,
        description="패턴 (스트라이프, 체크, 플로럴, 플레인 등. 불명확하면 null)"
    )
    item_style: Literal["casual", "formal", "sporty", "street", "vintage", "minimalist"] = Field(
        description="해당 아이템의 스타일. casual: 편안한 일상복, formal: 정장/드레스, sporty: 운동복, street: 스트릿, vintage: 복고풍, minimalist: 심플"
    )
    material: Optional[str] = Field(
        default=None,
        description="추정 소재 (코튼, 데님, 니트, 린넨 등. 불명확하면 null)"
    )


class FashionAnalysis(BaseModel):
    """패션 이미지 분석 결과"""

    # 성별 분류
    gender: Literal["male", "female", "unisex"] = Field(
        description="성별 카테고리. male: 남성 의류, female: 여성 의류, unisex: 남녀공용"
    )

    # 개별 아이템 상세 정보 (중첩 구조)
    items: List[ClothingItem] = Field(
        description="이미지에서 식별된 각 아이템의 상세 정보",
        min_length=1,
        max_length=5
    )

    # 전체 코디 스타일
    overall_style: Literal["casual", "formal", "sporty", "street", "vintage", "minimalist"] = Field(
        description="전체적인 코디 스타일. 단일 아이템이면 해당 아이템 스타일과 동일"
    )

    # 계절감
    season: Literal["spring", "summer", "fall", "winter", "all_season"] = Field(
        description="적합한 계절. spring: 봄, summer: 여름, fall: 가을, winter: 겨울, all_season: 사계절"
    )

    # 전체 설명
    description: str = Field(
        description="전체 코디/상품에 대한 간결한 설명 (100자 이내)",
        max_length=100
    )

print("패션 이미지 분석 스키마 정의 완료")

**실습 (2)**
- PydanticAI Agent를 생성하고 패션 이미지를 일괄 분석합니다.
- `BinaryContent`로 이미지를 전달하고, `output_type=FashionAnalysis`로 구조화된 결과를 받습니다.
- 결과를 JSON으로 저장하고 DataFrame으로 확인합니다.

In [ ]:
# =============================================================================
# 패션 이미지 분석 Agent 생성
# =============================================================================

# 시스템 프롬프트: 스키마 description에 없는 '행동 지시'만 작성합니다
# - 분류 기준(Enum 값 정의)은 이미 스키마에 포함되어 있으므로 중복하지 않습니다
fashion_system_prompt = """
당신은 패션 전문 데이터 분석가입니다.
의류 이미지를 보고 상품 정보를 정확하게 추출하세요.

[행동 규칙]
- 이미지에 실제로 보이는 아이템만 정확히 기재하세요.
- 여러 아이템이 보이면 모두 items 리스트에 포함하세요.
- 색상은 한글로 표기하세요 (블랙, 화이트, 네이비, 그레이, 베이지, 카키 등).
- 소재는 추정 가능한 경우에만 기재하세요.
- 코디 이미지의 overall_style은 전체 분위기로 판단하세요.
  예: 블레이저(formal) + 청바지(casual) => overall_style: casual
- description은 50-100자 이내로 작성하세요.
"""

# Agent 생성 - output_type으로 구조화된 출력을 강제합니다
fashion_agent = Agent(
    model_id,
    output_type=FashionAnalysis,
    system_prompt=fashion_system_prompt,
)

# 모델 설정 - 낮은 temperature로 일관된 분류 결과를 유도합니다
fashion_settings = GoogleModelSettings(temperature=0.2)

print("패션 분석 Agent 생성 완료")
print(f"  모델: {model_id}")
print(f"  output_type: FashionAnalysis")

In [ ]:
# =============================================================================
# 패션 이미지 일괄 분석
# =============================================================================

# 테스트 개수 설정 (None이면 전체 처리)
TEST_COUNT = 5

print("패션 이미지 분석 시작...")
print("=" * 80)

# 이미지 파일 목록 가져오기
fashion_images = sorted(
    list(FASHION_DIR.glob("*.jpg")) +
    list(FASHION_DIR.glob("*.jpeg")) +
    list(FASHION_DIR.glob("*.png"))
)
print(f"총 {len(fashion_images)}개 이미지 발견")

# 처리할 이미지 결정
if TEST_COUNT:
    fashion_images = fashion_images[:TEST_COUNT]
    print(f"테스트 모드: {TEST_COUNT}개만 처리")

# 분석 결과 저장
fashion_results = []

for i, image_path in enumerate(fashion_images, 1):
    print(f"\n[{i}/{len(fashion_images)}] 분석 중: {image_path.name}")

    # 이미지를 BinaryContent로 변환
    image_content = load_image_content(image_path)

    # PydanticAI Agent 실행 - 이미지 + 질문을 리스트로 전달
    result = await fashion_agent.run(
        [image_content, "이 패션 이미지를 분석하여 상품 정보를 추출해주세요."],
        model_settings=fashion_settings,
    )

    # result.output => FashionAnalysis 객체 (자동 파싱)
    output = result.output

    # 딕셔너리로 변환 + 파일명 추가
    row = {"image_file": image_path.name, **output.model_dump()}
    fashion_results.append(row)

    # 분석 결과 출력 (Literal 타입이므로 .value 없이 바로 사용)
    print(f"  - 성별: {output.gender}")
    print(f"  - 아이템 수: {len(output.items)}개")
    for idx, item in enumerate(output.items, 1):
        print(f"    [{idx}] {item.item_type}: {item.detailed_name}")
        print(f"        색상: {', '.join(item.colors)} | 스타일: {item.item_style}")
        if item.pattern:
            print(f"        패턴: {item.pattern}")
        if item.material:
            print(f"        소재: {item.material}")
    print(f"  - 전체 스타일: {output.overall_style}")
    print(f"  - 계절: {output.season}")
    print(f"  - 설명: {output.description}")

    time.sleep(API_DELAY)

print()
print("=" * 80)
print(f"패션 이미지 분석 완료: {len(fashion_results)}개 성공, {len(fashion_images) - len(fashion_results)}개 실패")

In [ ]:
# =============================================================================
# 패션 분석 결과 저장 및 DataFrame 변환
# =============================================================================

# 1. DataFrame 생성 - 정규화 구조 (1:N 관계)

# 1-1. 이미지 기본 정보 테이블
image_info = []
for result in fashion_results:
    image_info.append({
        'image_file': result['image_file'],
        'gender': result['gender'],
        'overall_style': result['overall_style'],
        'season': result['season'],
        'description': result['description'],
        'item_count': len(result['items'])
    })
image_df = pd.DataFrame(image_info)

# 1-2. 아이템 상세 정보 테이블
item_details = []
for result in fashion_results:
    for idx, item in enumerate(result['items'], 1):
        item_details.append({
            'image_file': result['image_file'],
            'item_number': idx,
            'item_type': item['item_type'],
            'detailed_name': item['detailed_name'],
            'colors': ', '.join(item['colors']) if isinstance(item['colors'], list) else item['colors'],
            'pattern': item['pattern'],
            'item_style': item['item_style'],
            'material': item['material']
        })
item_df = pd.DataFrame(item_details)

# 2. JSON 저장 (DataFrame의 to_json 활용)
json_output_path = OUTPUT_DIR / "fashion_analysis_results.json"
image_df.to_json(json_output_path, orient='records', force_ascii=False, indent=2)
print(f"JSON 저장: {json_output_path}")

# 3. 결과 출력
print()
print("[1. 이미지 기본 정보]")
print("=" * 80)
display(image_df)

print()
print("[2. 아이템 상세 정보]")
print("=" * 80)
display(item_df)

# 4. JOIN 예시 - 실무에서 정규화된 테이블을 결합하는 패턴입니다
print()
print("[JOIN 예시 - 이미지 정보 + 아이템 정보]")
print("=" * 80)
merged_df = item_df.merge(image_df, on='image_file', how='left')
display(merged_df)


**실습 (3)**
- 패션 분석 결과를 시각화하여 데이터의 분포와 패턴을 확인합니다.
- `seaborn`을 사용하여 성별/스타일/계절 분포와 크로스탭 분석을 수행합니다.

In [ ]:
# =============================================================================
# 패션 분석 결과 시각화
# =============================================================================

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("패션 이미지 분석 결과", fontsize=16, fontweight='bold')

# 1. 성별 분포
sns.countplot(data=image_df, x='gender', ax=axes[0, 0], palette='Set2')
axes[0, 0].set_title("성별 분포")
axes[0, 0].set_xlabel("성별")
axes[0, 0].set_ylabel("이미지 수")

# 2. 전체 스타일 분포
sns.countplot(data=image_df, x='overall_style', ax=axes[0, 1], palette='Set2')
axes[0, 1].set_title("전체 스타일 분포")
axes[0, 1].set_xlabel("스타일")
axes[0, 1].set_ylabel("이미지 수")
axes[0, 1].tick_params(axis='x', rotation=45)

# 3. 계절 분포
sns.countplot(data=image_df, x='season', ax=axes[1, 0], palette='Set2')
axes[1, 0].set_title("계절 분포")
axes[1, 0].set_xlabel("계절")
axes[1, 0].set_ylabel("이미지 수")

# 4. 아이템 타입 분포
sns.countplot(data=item_df, x='item_type', ax=axes[1, 1], palette='Set2')
axes[1, 1].set_title("아이템 타입 분포")
axes[1, 1].set_xlabel("아이템 타입")
axes[1, 1].set_ylabel("개수")
axes[1, 1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "fashion_analysis_charts.png", dpi=150, bbox_inches='tight', pad_inches=0.2)
plt.show()

# 크로스탭: 성별 x 스타일
print()
print("[크로스탭 분석: 성별 x 스타일]")
print("=" * 60)
cross_tab = pd.crosstab(image_df['gender'], image_df['overall_style'])
display(cross_tab)

# 통계 요약
print()
print("[분석 통계 요약]")
print(f"  총 이미지 수: {len(image_df)}개")
print(f"  총 아이템 수: {len(item_df)}개")
print(f"  평균 아이템/이미지: {item_df.groupby('image_file').size().mean():.1f}개")


### 2. 영수증 이미지 OCR 및 정형화

**비즈니스 목표**
- 수기/인쇄 영수증 자동 입력 시스템을 구축합니다.
- 회계 처리 시간을 99% 단축합니다.
- 입력 오류를 제로화합니다.

**활용 사례**
- 경비 처리 자동화
- 회계 시스템 자동 연동
- 지출 분석 및 리포트

#### 2-1. 영수증 분석 스키마 설계

**영수증 분석 스키마 설계 프로세스**

**1. 비즈니스 질문부터 시작**

**Q: 영수증 사진으로부터 무엇을 얻고 싶은가?**

A: 영수증 이미지를 분석해서 가맹점 정보, 구매 상품 목록, 금액 세부사항, 결제 정보를 자동으로 추출하고 데이터 품질을 평가하고 싶습니다.

**2. 필요한 정보 나열**

- **가맹점 정보**: 상호명, 주소, 전화번호
- **거래 정보**: 거래 일시, 영수증 번호
- **구매 상품 상세**: 상품명, 수량, 단가, 합계
- **금액 정보**: 소계, 세금, 봉사료, 할인, 최종 결제 금액
- **결제 방식**: 현금, 카드, 복합결제
- **OCR 품질**: 인식률, 데이터 신뢰도

**3. 데이터 타입 결정**

| 정보 | 타입 | 이유 |
|------|------|------|
| 결제 수단 | Enum (PaymentMethod) | 4가지 고정 옵션 (현금/카드/복합/불명) |
| 가맹점명 | Optional[str] | 인식 불가능할 수 있음 |
| 거래일시 | Optional[str] | YYYY-MM-DD HH:MM:SS 형식 |
| 상품 목록 | List[ReceiptItem] | 최소 1개 이상 (중첩 구조) |
| 금액 필드 | float | 0 이상의 실수값 |
| OCR 신뢰도 | float | 0.0~1.0 범위 제한 |

**4. 구조 설계 포인트**

- **중첩 구조 활용**: `ReceiptItem`을 별도 클래스로 분리하여 각 상품의 상세 정보를 체계적으로 관리합니다.
- **Optional 필드 활용**: 영수증 상태에 따라 없을 수 있는 정보는 모두 Optional로 처리합니다.
- **제약 조건 설정**: quantity >= 1, 금액 >= 0, 신뢰도 0.0~1.0

**실습 (1)**
- 영수증 분석을 위한 Pydantic 스키마를 정의합니다.
- `ReceiptItem` 중첩 구조로 개별 상품 정보를 관리하고, `PaymentMethod` Enum으로 결제 수단을 표준화합니다.

In [ ]:
# =============================================================================
# 영수증 분석 스키마 정의
# =============================================================================

class ReceiptItem(BaseModel):
    """개별 상품 정보 (중첩 구조)"""
    item_name: str = Field(description="상품명 (띄어쓰기 보정)")
    quantity: int = Field(ge=1, description="수량")
    unit_price: float = Field(ge=0, description="단가 (숫자만, 통화 기호 제거)")
    total_price: float = Field(ge=0, description="해당 상품 합계 금액")


class ReceiptAnalysis(BaseModel):
    """영수증 분석 결과"""

    # 가맹점 정보
    store_name: Optional[str] = Field(
        default=None,
        description="가맹점명/상호 (읽을 수 없으면 null)"
    )
    store_address: Optional[str] = Field(
        default=None,
        description="주소 (읽을 수 있는 경우만)"
    )
    store_tel: Optional[str] = Field(
        default=None,
        description="전화번호 (읽을 수 있는 경우만)"
    )

    # 거래 정보
    transaction_date: Optional[str] = Field(
        default=None,
        description="거래 일시 (YYYY-MM-DD HH:MM:SS 형식. 시간 없으면 YYYY-MM-DD)"
    )
    receipt_number: Optional[str] = Field(
        default=None,
        description="영수증 번호/거래번호 (있는 경우만)"
    )

    # 상품 목록 (중첩 구조)
    items: List[ReceiptItem] = Field(
        description="구매 상품 목록",
        min_length=1
    )

    # 금액 정보
    subtotal: float = Field(ge=0, description="소계 (상품 금액 합계)")
    tax: Optional[float] = Field(default=0, description="세금 (VAT, Tax 등)")
    service_charge: Optional[float] = Field(default=0, description="봉사료 (있는 경우만)")
    discount: Optional[float] = Field(default=0, description="할인 금액 (있는 경우만)")
    total_amount: float = Field(ge=0, description="최종 결제 금액")

    # 결제 정보
    payment_method: Literal["cash", "card", "mixed", "unknown"] = Field(
        description="결제 수단. cash: 현금, card: 카드, mixed: 복합 결제, unknown: 불명"
    )
    cash_received: Optional[float] = Field(default=None, description="받은 현금 (현금 결제 시)")
    change: Optional[float] = Field(default=None, description="거스름돈 (현금 결제 시)")

    # 품질 평가
    ocr_confidence: float = Field(
        ge=0.0, le=1.0,
        description="OCR 신뢰도 (0.0~1.0). 0.9+: 모든 정보 명확, 0.7~0.9: 대부분 판독, 0.5~0.7: 일부 불명확"
    )
    data_quality: Literal["excellent", "good", "fair", "poor"] = Field(
        description="데이터 품질. excellent: 필수 정보 완벽, good: 대부분 확보, fair: 일부만 확보, poor: 대부분 누락"
    )

print("영수증 분석 스키마 정의 완료")

**실습 (2)**
- PydanticAI Agent를 생성하고 영수증 이미지를 일괄 분석합니다.
- `BinaryContent`로 이미지를 전달하고, `output_type=ReceiptAnalysis`로 구조화된 결과를 받습니다.
- 결과를 JSON으로 저장하고 DataFrame으로 확인합니다.

In [ ]:
# =============================================================================
# 영수증 분석 Agent 생성
# =============================================================================

# 시스템 프롬프트: 스키마 description에 없는 '행동 지시'만 작성합니다
receipt_system_prompt = """
당신은 영수증 OCR 및 데이터 추출 전문가입니다.
영수증 이미지에서 정보를 정확하게 추출하여 구조화하세요.

[행동 규칙]
- 모든 금액은 숫자만 추출하세요 (쉼표, 통화 기호 제거).
- 소수점이 있으면 그대로 유지하세요 (예: 29.50).
- 상품명은 띄어쓰기를 보정하세요.
- 읽을 수 없는 정보는 null로 처리하세요.
- 금액 검증: subtotal + tax + service_charge - discount 가 total_amount과 일치해야 합니다.
- 흐릿하거나 구겨진 영수증도 최대한 판독하세요.
- 불확실한 경우 ocr_confidence를 낮게 설정하세요.
- 금액 계산이 맞지 않으면 data_quality를 낮추세요.
"""

# Agent 생성
receipt_agent = Agent(
    model_id,
    output_type=ReceiptAnalysis,
    system_prompt=receipt_system_prompt,
)

# 모델 설정 - 매우 낮은 temperature로 정확성 우선
receipt_settings = GoogleModelSettings(temperature=0.1)

print("영수증 분석 Agent 생성 완료")
print(f"  모델: {model_id}")
print(f"  output_type: ReceiptAnalysis")

In [ ]:
# =============================================================================
# 영수증 이미지 일괄 분석
# =============================================================================

# 테스트 개수 설정 (None이면 전체 처리)
TEST_COUNT = 10

print("영수증 이미지 분석 시작...")
print("=" * 80)

# 이미지 파일 목록 가져오기
receipt_images = sorted(
    list(RECEIPT_DIR.glob("*.jpg")) +
    list(RECEIPT_DIR.glob("*.png")) +
    list(RECEIPT_DIR.glob("*.jpeg"))
)
print(f"총 {len(receipt_images)}개 영수증 발견")

# 처리할 이미지 결정
if TEST_COUNT:
    receipt_images = receipt_images[:TEST_COUNT]
    print(f"테스트 모드: {TEST_COUNT}개만 처리")

# 분석 결과 저장
receipt_results = []

for i, image_path in enumerate(receipt_images, 1):
    print(f"\n[{i}/{len(receipt_images)}] 분석 중: {image_path.name}")

    # 이미지를 BinaryContent로 변환
    image_content = load_image_content(image_path)

    # PydanticAI Agent 실행 - 이미지 + 질문을 리스트로 전달
    prompt = "이 영수증 이미지를 분석하여 모든 정보를 추출해주세요."
    result = await receipt_agent.run(
        [image_content, prompt],
        model_settings=receipt_settings,
    )

    # result.output => ReceiptAnalysis 객체 (자동 파싱)
    output = result.output

    # 딕셔너리로 변환 + 파일명 추가
    row = {"image_file": image_path.name, **output.model_dump()}
    receipt_results.append(row)

    # 분석 결과 출력 (Literal 타입이므로 .value 없이 바로 사용)
    print(f"  - 가맹점: {output.store_name or '미확인'}")
    print(f"  - 거래일시: {output.transaction_date or '미확인'}")
    print(f"  - 상품 수: {len(output.items)}개")
    print(f"  - 총 금액: ${output.total_amount:,.2f}")
    print(f"  - 결제: {output.payment_method}")
    print(f"  - 신뢰도: {output.ocr_confidence:.2f} ({output.data_quality})")

    time.sleep(API_DELAY)

print()
print("=" * 80)
print(f"영수증 분석 완료: {len(receipt_results)}개 성공, {len(receipt_images) - len(receipt_results)}개 실패")

In [ ]:
# =============================================================================
# 영수증 분석 결과 저장 및 DataFrame 변환
# =============================================================================

# 1. DataFrame 생성 - 정규화 구조

# 1-1. 영수증 기본 정보 테이블
receipt_info = []
for result in receipt_results:
    receipt_info.append({
        'image_file': result['image_file'],
        'store_name': result['store_name'],
        'transaction_date': result['transaction_date'],
        'subtotal': result['subtotal'],
        'tax': result['tax'],
        'service_charge': result['service_charge'],
        'discount': result['discount'],
        'total_amount': result['total_amount'],
        'payment_method': result['payment_method'],
        'ocr_confidence': result['ocr_confidence'],
        'data_quality': result['data_quality'],
        'item_count': len(result['items'])
    })
receipt_df = pd.DataFrame(receipt_info)

# 1-2. 상품 상세 정보 테이블
item_details = []
for result in receipt_results:
    for idx, item in enumerate(result['items'], 1):
        item_details.append({
            'image_file': result['image_file'],
            'item_number': idx,
            'item_name': item['item_name'],
            'quantity': item['quantity'],
            'unit_price': item['unit_price'],
            'total_price': item['total_price']
        })
receipt_item_df = pd.DataFrame(item_details)

# 2. JSON 저장 (DataFrame의 to_json 활용)
json_output_path = OUTPUT_DIR / "receipt_analysis_results.json"
receipt_df.to_json(json_output_path, orient='records', force_ascii=False, indent=2)
print(f"JSON 저장: {json_output_path}")

# 3. 결과 출력
print()
print("[1. 영수증 기본 정보]")
print("=" * 80)
display(receipt_df)

print()
print("[2. 상품 상세 정보]")
print("=" * 80)
display(receipt_item_df)

# 4. JOIN 예시
print()
print("[JOIN 예시 - 영수증 정보 + 상품 정보]")
print("=" * 80)
merged_df = receipt_item_df.merge(receipt_df, on='image_file', how='left')
display(merged_df.head(10))


**실습 (3)**
- 영수증 분석 결과를 시각화하여 데이터의 분포를 확인합니다.
- **금액 일관성 검증**: LLM이 추출한 subtotal + tax - discount이 total_amount과 일치하는지 코드로 확인합니다.
- 이 검증은 실무에서 LLM 출력을 신뢰하기 전에 반드시 수행해야 하는 과정입니다.

In [ ]:
# =============================================================================
# 영수증 분석 결과 시각화 및 데이터 품질 검증
# =============================================================================

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("영수증 분석 결과", fontsize=16, fontweight='bold')

# 1. 결제 수단 분포
sns.countplot(data=receipt_df, x='payment_method', ax=axes[0, 0], palette='Set2')
axes[0, 0].set_title("결제 수단 분포")
axes[0, 0].set_xlabel("결제 수단")
axes[0, 0].set_ylabel("영수증 수")

# 2. 결제 금액 분포
sns.histplot(data=receipt_df, x='total_amount', bins=10, ax=axes[0, 1], color='steelblue')
axes[0, 1].set_title("결제 금액 분포")
axes[0, 1].set_xlabel("금액 ($)")
axes[0, 1].set_ylabel("영수증 수")

# 3. OCR 신뢰도 분포
sns.histplot(data=receipt_df, x='ocr_confidence', bins=10, ax=axes[1, 0], color='teal')
axes[1, 0].set_title("OCR 신뢰도 분포")
axes[1, 0].set_xlabel("신뢰도")
axes[1, 0].set_ylabel("영수증 수")
axes[1, 0].axvline(x=0.7, color='red', linestyle='--', alpha=0.7, label='기준선 (0.7)')
axes[1, 0].legend()

# 4. 데이터 품질 분포
quality_order = [q for q in ['excellent', 'good', 'fair', 'poor'] if q in receipt_df['data_quality'].values]
sns.countplot(data=receipt_df, x='data_quality', ax=axes[1, 1], palette='RdYlGn_r',
              order=quality_order)
axes[1, 1].set_title("데이터 품질 분포")
axes[1, 1].set_xlabel("품질 등급")
axes[1, 1].set_ylabel("영수증 수")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "receipt_analysis_charts.png", dpi=150, bbox_inches='tight', pad_inches=0.2)
plt.show()

# =========================================================================
# 금액 일관성 검증
# - LLM이 추출한 금액이 논리적으로 맞는지 코드로 확인합니다
# - 실무에서 LLM 출력을 DB에 저장하기 전 필수 검증 단계입니다
# =========================================================================
print()
print("[금액 일관성 검증]")
print("=" * 60)
print("검증 공식: subtotal + tax + service_charge - discount = total_amount")
print()

receipt_df['expected_total'] = (
    receipt_df['subtotal']
    + receipt_df['tax'].fillna(0)
    + receipt_df['service_charge'].fillna(0)
    - receipt_df['discount'].fillna(0)
)
receipt_df['amount_diff'] = abs(receipt_df['expected_total'] - receipt_df['total_amount'])
receipt_df['amount_match'] = receipt_df['amount_diff'] < 1.0  # $1 이내 허용

# 검증 결과 테이블 출력
verify_df = receipt_df[['image_file', 'subtotal', 'tax', 'discount', 'total_amount', 'expected_total', 'amount_diff', 'amount_match']]
display(verify_df)

# 요약 통계
match_rate = receipt_df['amount_match'].mean() * 100
print()
print(f"  금액 일치율: {match_rate:.0f}% ({receipt_df['amount_match'].sum()}/{len(receipt_df)}개)")
print(f"  평균 오차: ${receipt_df['amount_diff'].mean():.2f}")

if match_rate < 100:
    print()
    print("  [불일치 영수증]")
    mismatch = receipt_df[~receipt_df['amount_match']]
    for _, row in mismatch.iterrows():
        print(f"    - {row['image_file']}: 예상 ${row['expected_total']:.2f} vs 실제 ${row['total_amount']:.2f} (차이: ${row['amount_diff']:.2f})")

# 전체 통계 요약
print()
print("[전체 통계 요약]")
print("=" * 60)
print(f"  총 영수증 수: {len(receipt_df)}개")
print(f"  총 상품 수: {len(receipt_item_df)}개")
print(f"  평균 상품/영수증: {receipt_item_df.groupby('image_file').size().mean():.1f}개")
print(f"  총 거래액: ${receipt_df['total_amount'].sum():,.2f}")
print(f"  평균 구매액: ${receipt_df['total_amount'].mean():,.2f}")
print(f"  평균 OCR 신뢰도: {receipt_df['ocr_confidence'].mean():.2f}")
